# Dự đoán và đánh giá cuối ensemble C30

Mục 1–4 nạp artifact fit train để kiểm tra validation. Mục 5–7 giữ nguyên cấu hình đã chốt, fit lại trên train + validation đủ history và đánh giá test.

Kết quả test lưu riêng; lần chạy sau đọc lại kết quả, không tự fit hoặc chấm lại. Dữ liệu demo MnDOT + weather cần chuẩn hóa riêng.


## 1. Nạp mô hình

`predict.py` nằm ở thư mục gốc để dùng chung với notebook và Streamlit. Artifact gồm ba mô hình, trọng số, thứ tự cột và thống kê điền thiếu đã học. Muốn dùng mô hình refit cuối, thay `MODEL_PATH` bằng artifact có cùng định dạng.


In [4]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'predict.py').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from predict import DEFAULT_MODEL, load_model, read_hourly, prepare_features, predict_hours

MODEL_PATH = DEFAULT_MODEL
model = load_model(MODEL_PATH)
print('Mô hình:', model['spec']['model_id'])
print('Phạm vi:', model['scope'])
print('Fit đến:', model['train_end'])


Mô hình: C30_RF25_XGB25_ExtraTrees50
Phạm vi: train-only fit for validation interpretation
Fit đến: 2017-05-10 04:00:00


## 2. Đọc dữ liệu đầu vào và ngữ cảnh quá khứ

CSV cần một dòng mỗi giờ, có `date_time`, `temp` (K), `rain_1h`, `snow_1h` (mm), `clouds_all` (%), `weather_main`, `is_holiday` và lưu lượng quá khứ `traffic_volume`. Nhãn weather phải theo quy tắc của C30.

Chỉ đọc các giá trị đến hết validation. Các giờ trước dùng tạo lag và mean; giờ đang dự đoán không tham gia history. Không lấy thống kê điền thiếu mới từ dữ liệu đầu vào.


In [5]:
data_path = ROOT / 'data/preprocessed/hourly_unfilled.csv'
n_total = len(pd.read_csv(data_path, usecols=['date_time']))
hourly = read_hourly(data_path, nrows=n_total * 85 // 100)
reference = pd.read_csv(ROOT / 'experiments/22-11-2026_Huy_C30_MAE_Diversity/outputs/validation_predictions.csv',
                        parse_dates=['date_time'], float_precision='round_trip')
times = reference.date_time
print(f'Dữ liệu ngữ cảnh: {len(hourly):,} giờ; cần dự đoán {len(times):,} giờ validation.')


Dữ liệu ngữ cảnh: 34,488 giờ; cần dự đoán 6,050 giờ validation.


## 3. Dự đoán và kiểm tra lại đường inference

Hàm `predict_hours` tạo C30 bằng quy tắc cũ, sau đó kết hợp RF 25%, XGBoost 25%, ExtraTrees 50%. Giờ thiếu history giữ dự đoán trống và có thông báo. Giá trị thực tế chỉ được ghép sau khi dự đoán để tính sai số.

Đối chiếu với dự đoán validation đã lưu để xác nhận `predict.py` giữ đúng mô hình và cách chuẩn bị đầu vào.


In [6]:
result = predict_hours(hourly, times, model)
assert result.prediction.notna().all()
np.testing.assert_allclose(result.prediction, reference['Selected by OOF'], rtol=0, atol=1e-5)
result['actual'] = reference.y_true.to_numpy()
result['absolute_error'] = (result.prediction - result.actual).abs()
mae = result.absolute_error.mean()

OUT = ROOT / 'notebooks/validation/outputs/predict_C30'
OUT.mkdir(parents=True, exist_ok=True)
result.to_csv(OUT / 'validation_predictions.csv', index=False)
record = {'artifact': str(MODEL_PATH), 'scope': model['scope'], 'n': len(result),
          'MAE': float(mae), 'fit_performed': False, 'test_evaluated': False,
          'max_difference_from_saved_predictions': float((result.prediction - reference['Selected by OOF']).abs().max())}
_ = (OUT / 'prediction_check.json').write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding='utf-8')
display(result.head().round({'prediction': 2, 'absolute_error': 2}))
display(Markdown(f'Đường predict tái lập đúng kết quả validation: **{len(result):,} giờ**, MAE **{mae:.4f} xe/giờ**. Đây là kiểm tra inference trên cùng dữ liệu, chưa phải đánh giá mới.'))


,date_time,prediction,status,actual,absolute_error
0,2017-05-10 05:00:00,2981.84,Đủ đầu vào,2920,61.84
1,2017-05-10 06:00:00,5976.20,Đủ đầu vào,5988,11.80
2,2017-05-10 07:00:00,6545.84,Đủ đầu vào,6739,193.16
3,2017-05-10 08:00:00,5849.10,Đủ đầu vào,5847,2.10
4,2017-05-10 09:00:00,5214.39,Đủ đầu vào,5062,152.39


Đường predict tái lập đúng kết quả validation: **6,050 giờ**, MAE **190.1900 xe/giờ**. Đây là kiểm tra inference trên cùng dữ liệu, chưa phải đánh giá mới.

## 4. Chọn một giờ để minh họa

Thay `chosen_time` bằng một giờ hợp lệ sau thời điểm fit. Bảng đầu tiên cho xem đầu vào C30; bảng tiếp theo là dự đoán. Thực tế chỉ dùng để đối chiếu sau đó.

Nếu không có lưu lượng thật tại giờ cần dự đoán, có thể để `traffic_volume` của giờ đó là NaN; các giờ quá khứ cần cho history vẫn phải có quan sát.


In [7]:
chosen_time = times.iloc[0]
inputs = prepare_features(hourly, [chosen_time], model)
example = predict_hours(hourly, [chosen_time], model)
display(inputs)
display(example.round({'prediction': 2}))

time_mask = hourly.date_time.eq(chosen_time)
without_target = hourly.copy()
without_target.loc[time_mask, 'traffic_volume'] = np.nan
check = predict_hours(without_target, [chosen_time], model)
np.testing.assert_allclose(example.prediction, check.prediction, rtol=0, atol=1e-8)
display(Markdown('Bỏ lưu lượng thật của giờ đang dự đoán không làm thay đổi kết quả. Mô hình chỉ sử dụng lưu lượng quá khứ cho history.'))


,hour,day_of_week,month,is_weekend,is_rush_hour,hour_sin,hour_cos,day_of_week_sin,day_of_week_cos,month_sin,...,weather_haze,weather_mist,weather_rain,weather_smoke,weather_snow,weather_squall,weather_thunderstorm,lag_24h,lag_168h,mean_24h
date_time,,,,,,,,,,,,,,,,,,,,,
2017-05-10 05:00:00,5.0,2.0,5.0,0.0,0.0,0.965926,0.258819,0.974928,-0.222521,0.866025,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2921.0,2970.0,3707.0


,date_time,prediction,status
0,2017-05-10 05:00:00,2981.84,Đủ đầu vào


Bỏ lưu lượng thật của giờ đang dự đoán không làm thay đổi kết quả. Mô hình chỉ sử dụng lưu lượng quá khứ cho history.

In [8]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        str(ROOT / "predict.py"),
        "--input", str(ROOT / "data/preprocessed/hourly_unfilled.csv"),
        "--at", "2017-05-10 05:00:00",
    ],
    check=True,
)

CompletedProcess(args=['e:\\Project\\SGU_HocMay_2026-2027\\.venv\\Scripts\\python.exe', 'e:\\Project\\SGU_HocMay_2026-2027\\metro_interstate_traffic_volume\\predict.py', '--input', 'e:\\Project\\SGU_HocMay_2026-2027\\metro_interstate_traffic_volume\\data\\preprocessed\\hourly_unfilled.csv', '--at', '2017-05-10 05:00:00'], returncode=0)

In [9]:
subprocess.run(
    [sys.executable, str(ROOT / 'predict.py'), '--input', str(data_path),
     '--at', '2017-05-10 05:00:00'],
    check=True,
)


CompletedProcess(args=['e:\\Project\\SGU_HocMay_2026-2027\\.venv\\Scripts\\python.exe', 'e:\\Project\\SGU_HocMay_2026-2027\\metro_interstate_traffic_volume\\predict.py', '--input', 'e:\\Project\\SGU_HocMay_2026-2027\\metro_interstate_traffic_volume\\data\\preprocessed\\hourly_unfilled.csv', '--at', '2017-05-10 05:00:00'], returncode=0)

## 5. Fit cuối trên train + validation

Giữ nguyên tham số RF/XGBoost/ExtraTrees và trọng số 25%/25%/50%. Hàm `fit_final` chỉ học mô hình và thống kê điền thiếu trên cohort đủ history thuộc 85% dữ liệu đầu. Lưu kế hoạch gồm hash dữ liệu, tham số, code và phiên bản trước khi chấm test. Nếu artifact đã có và đúng kế hoạch, chỉ nạp lại.

Phần kiểm tra validation bên trên vẫn dùng artifact cũ. Artifact cuối fit thêm validation chỉ dùng dự đoán giai đoạn sau mốc fit.


In [10]:
from predict import fit_final, evaluate_test_once

final_model, all_hourly, final_plan = fit_final()
FINAL = ROOT / 'notebooks/validation/outputs/final_C30'
FINAL_MODEL = FINAL / 'final_model.joblib'
display(pd.DataFrame([{'scope': 'fit train+validation', 'n': final_plan['n_fit'],
                       'end': final_model['train_end']},
                      {'scope': 'test trước lọc history', 'n': final_plan['n_test_total']},
                      {'scope': 'test đủ history', 'n': final_plan['n_test_history']}]))


Nạp lại mô hình fit cuối đã lưu.


,scope,n,end
0,fit train+validation,28553,2018-01-19 14:00:00
1,test trước lọc history,6087,NaN
2,test đủ history,6052,NaN


## 6. Đánh giá test một lần

Dự đoán toàn bộ giờ test; chỉ chấm những giờ đủ history và ghi số giờ bị loại. Kịch bản là dự đoán tuần tự: các quan sát test trước giờ đang dự đoán được dùng tạo lag/mean, còn target hiện tại không nằm trong đầu vào. Thời tiết cùng giờ được giả định sẵn có.

Lượt đầu tạo dấu bắt đầu và lưu kết quả. Những lần sau kiểm tra hash rồi đọc lại kết quả, không dự đoán hoặc chấm lại. Nếu lượt đầu bị ngắt, hàm dừng để kiểm tra trước khi thử tiếp. Không đổi tham số hoặc trọng số theo điểm test.


In [11]:
test_result, test_metrics = evaluate_test_once(all_hourly, final_model)
display(test_metrics.round({'MAE': 3, 'RMSE': 3, 'R2': 6}))
display(test_result.head().round({'prediction': 2, 'absolute_error': 2}))


Đọc lại kết quả test đã lưu; không dự đoán hoặc chấm lại.


,model,config,n_fit,n_test_total,n,n_excluded,start,end,MAE,RMSE,R2
0,C30_RF25_XGB25_ExtraTrees50,C30,28553,6087,6052,35,2018-01-19 15:00:00,2018-09-30 23:00:00,178.208,295.528,0.977587


,date_time,prediction,status,actual,absolute_error
0,2018-01-19 15:00:00,5758.30,Đủ đầu vào,5810,51.70
1,2018-01-19 16:00:00,6294.31,Đủ đầu vào,6305,10.69
2,2018-01-19 17:00:00,5870.49,Đủ đầu vào,5657,213.49
3,2018-01-19 18:00:00,4654.15,Đủ đầu vào,4503,151.15
4,2018-01-19 19:00:00,3560.08,Đủ đầu vào,3516,44.08


## 7. Nhận xét và dùng mô hình cuối

MAE và RMSE có đơn vị xe/giờ; R² không phải tỷ lệ dự đoán đúng. Điểm test thuộc giai đoạn khác validation, nên chênh lệch giữa hai điểm chưa đủ để kết luận overfitting. Giữ nguyên kết quả đã đo và dùng artifact cuối cho demo sau khi chuẩn hóa dữ liệu ngoài UCI.


In [12]:
r = test_metrics.iloc[0]
note = (f'Mô hình fit trên {int(r.n_fit):,} giờ train + validation. '
        f'Chấm {int(r.n):,}/{int(r.n_test_total):,} giờ test, loại {int(r.n_excluded):,} giờ thiếu history. '
        f'MAE = {r.MAE:.3f}, RMSE = {r.RMSE:.3f} xe/giờ; R² = {r.R2:.6f}. '
        'Không dùng điểm test để chọn lại mô hình; đây là kết quả theo kịch bản tuần tự đã công bố.')
display(Markdown(note))
_ = (FINAL / 'test_observations.md').write_text(note, encoding='utf-8')
print('Artifact dùng cho dự đoán mới:', FINAL_MODEL)


Mô hình fit trên 28,553 giờ train + validation. Chấm 6,052/6,087 giờ test, loại 35 giờ thiếu history. MAE = 178.208, RMSE = 295.528 xe/giờ; R² = 0.977587. Không dùng điểm test để chọn lại mô hình; đây là kết quả theo kịch bản tuần tự đã công bố.

Artifact dùng cho dự đoán mới: e:\Project\SGU_HocMay_2026-2027\metro_interstate_traffic_volume\notebooks\validation\outputs\final_C30\final_model.joblib


## 8. Dữ liệu demo ngoài UCI

Sau khi có CSV MnDOT + weather đã chuẩn hóa, thay nguồn đọc và truyền các timestamp cần dự đoán vào cùng hàm. Giữ đủ ngữ cảnh quá khứ. Không thay dữ liệu bằng weather giả hoặc điền median cho lag.

Có thể chạy từ thư mục gốc:

```bash
python predict.py --input data/preprocessed/hourly_unfilled.csv --at "2017-05-10 05:00:00"
```

Sau mục 7, truyền `--model notebooks/validation/outputs/final_C30/final_model.joblib` để dùng mô hình cuối. Ví dụ cũ bên trên vẫn dùng artifact train-only mặc định. Mô hình cuối chỉ dự đoán các giờ sau thời điểm fit của nó. Dữ liệu demo mới không được dùng để fit lại mô hình trong notebook này.
